# 02 — Preprocessing: Model Rekomendasi Latihan

**Stack SOTA 2024-2025:**
- **RobustScaler** untuk fitur numerik (handle outliers BMI/Calories)
- **Feature engineering:** BMR (Mifflin-St Jeor), TDEE, FFMI, interaction features
- **SMOTEENN** untuk class imbalance (kombinasi SMOTE + EditedNN)
- **iterative-stratification** untuk multi-label CV split
- **Augmentasi 7-day** dari 2,773 → ~19,400 baris

**Input:** 2 dataset combined (real 973 + synthetic 1,800)

**Output:**
- `output/preprocessed/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`
- `output/preprocessed/scaler.pkl` (RobustScaler — untuk inference)
- `output/preprocessed/metadata.json`

**Riset acuan:**
- i-MRI 2024: RobustScaler val_acc 0.73 > StandardScaler 0.71
- Springer AI Review 2024: SMOTEENN paling robust untuk medical data
- Nature Sci Reports 2025: interaction features +3-4pp AUC

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours

np.random.seed(42)
os.makedirs('output/preprocessed', exist_ok=True)

# ── Load Combined Dataset ──
PATH_REAL      = '../../dataset/Model_rekomendasi_Pelatihan/gym_member_exercise_dataset/gym_members_exercise_tracking.csv'
PATH_SYNTHETIC = '../../dataset/Model_Adaptif_Perencanaan_Ulang/Fitness Tracker Dataset/gym_members_exercise_tracking_synthetic_data.csv'

df_real      = pd.read_csv(PATH_REAL)
df_synthetic = pd.read_csv(PATH_SYNTHETIC)
df = pd.concat([df_real, df_synthetic], ignore_index=True)
print(f'Combined: {df.shape}')

In [ ]:
# ── 1. Feature Engineering (Riset Nature 2025: +3-4pp AUC) ──

def engineer_features(df):
    df = df.copy()
    
    # ── Basic encoding ──
    df['gender_enc'] = df['Gender'].map({'Male': 1, 'Female': 0}).fillna(2).astype(int)
    
    # ── BMR (Mifflin-St Jeor) ──
    df['height_cm'] = df['Height (m)'] * 100
    df['bmr'] = (10 * df['Weight (kg)'] +
                 6.25 * df['height_cm'] -
                 5 * df['Age'] +
                 np.where(df['gender_enc'] == 1, 5, -161))
    
    # ── TDEE (BMR × activity factor) ──
    # Activity factor based on workout frequency
    activity_map = {1: 1.2, 2: 1.375, 3: 1.55, 4: 1.725, 5: 1.9}
    df['activity_factor'] = df['Workout_Frequency (days/week)'].map(activity_map).fillna(1.55)
    df['tdee'] = df['bmr'] * df['activity_factor']
    
    # ── FFMI (Fat-Free Mass Index) ──
    df['lean_mass_kg'] = df['Weight (kg)'] * (1 - df['Fat_Percentage'] / 100)
    df['ffmi'] = df['lean_mass_kg'] / (df['Height (m)'] ** 2)
    
    # ── BMI category ──
    def bmi_cat(b):
        if b < 18.5: return 0  # Underweight
        if b < 25:   return 1  # Normal
        if b < 30:   return 2  # Overweight
        return 3                # Obese
    df['bmi_cat'] = df['BMI'].apply(bmi_cat)
    
    # ── Age band ──
    df['age_band'] = pd.cut(df['Age'], bins=[0, 25, 35, 50, 120],
                            labels=[0, 1, 2, 3]).astype(int)
    
    # ── Interaction features (Nature 2025: improves AUC 3-4pp) ──
    df['bmi_x_age'] = df['BMI'] * df['Age']
    df['activity_x_duration'] = df['Workout_Frequency (days/week)'] * df['Session_Duration (hours)']
    df['bpm_intensity_ratio'] = df['Avg_BPM'] / df['Max_BPM']
    df['calorie_efficiency'] = df['Calories_Burned'] / df['Session_Duration (hours)']
    
    # ── Target encoding ──
    workout_map = {'Yoga': 0, 'Cardio': 1, 'Strength': 2, 'HIIT': 3}
    df['workout_type_label'] = df['Workout_Type'].map(workout_map).fillna(1).astype(int)
    df['intensity_label'] = df['Experience_Level'].astype(int)  # 1,2,3
    
    # ── Mode (semua dataset = gym, default 1) ──
    df['mode'] = 1
    df['has_injury'] = 0
    df['has_chronic'] = 0
    df['days_per_week'] = df['Workout_Frequency (days/week)']
    df['session_minutes'] = (df['Session_Duration (hours)'] * 60).round().astype(int)
    
    return df

df_feat = engineer_features(df)
print(f'Feature engineering selesai. Total kolom: {len(df_feat.columns)}')
print('\nFitur baru hasil engineering:')
new_features = ['bmr', 'tdee', 'ffmi', 'bmi_x_age', 'activity_x_duration',
                'bpm_intensity_ratio', 'calorie_efficiency']
df_feat[new_features].describe().round(2)

In [ ]:
# ── 2. Augmentasi 7-Day Expansion ──
# 1 user → 7 hari (5-6 hari latihan + sisanya REST)

FEATURE_COLS = ['BMI', 'bmi_cat', 'gender_enc', 'Age', 'age_band',
                'Experience_Level', 'mode', 'days_per_week', 'session_minutes',
                'has_injury', 'has_chronic',
                'bmr', 'tdee', 'ffmi',
                'bmi_x_age', 'activity_x_duration', 'bpm_intensity_ratio',
                'calorie_efficiency']

rows = []
for _, user in df_feat.iterrows():
    days_per_week = int(user['days_per_week'])
    days_per_week = max(1, min(days_per_week, 6))
    
    # Pilih hari latihan secara random tapi reproducible
    train_days = set(np.random.choice(range(7), size=days_per_week, replace=False))
    
    for day_idx in range(7):
        row = {col: user[col] for col in FEATURE_COLS if col in user.index}
        row['day_index'] = day_idx
        row['is_first_day'] = int(day_idx == 0)
        
        if day_idx not in train_days:
            # REST day
            row['workout_type_label'] = 4  # REST = label 4
            row['intensity_label'] = 1     # LOW for rest
        else:
            # Active day — small noise pada numeric features
            row['BMI'] = float(user['BMI']) + np.random.normal(0, 0.2)
            row['Age'] = int(user['Age']) + np.random.randint(-1, 2)
            row['workout_type_label'] = int(user['workout_type_label'])
            row['intensity_label'] = int(user['intensity_label'])
        
        rows.append(row)

df_aug = pd.DataFrame(rows)

# Recompute interaction features after noise
df_aug['bmi_cat'] = df_aug['BMI'].apply(
    lambda b: 0 if b<18.5 else 1 if b<25 else 2 if b<30 else 3
)
df_aug['age_band'] = pd.cut(df_aug['Age'], bins=[0, 25, 35, 50, 120],
                            labels=[0, 1, 2, 3]).astype(int)
df_aug['bmi_x_age'] = df_aug['BMI'] * df_aug['Age']

print(f'Augmented: {df_aug.shape}')
print(f'\nWorkout type distribution:')
print(df_aug['workout_type_label'].value_counts())
print(f'\nIntensity distribution:')
print(df_aug['intensity_label'].value_counts())

In [ ]:
# ── 3. Final Feature Set ──
FEATURE_FINAL = ['BMI', 'bmi_cat', 'gender_enc', 'Age', 'age_band',
                 'Experience_Level', 'mode', 'days_per_week', 'session_minutes',
                 'day_index', 'is_first_day', 'has_injury', 'has_chronic',
                 'bmr', 'tdee', 'ffmi',
                 'bmi_x_age', 'activity_x_duration', 'bpm_intensity_ratio',
                 'calorie_efficiency']

X = df_aug[FEATURE_FINAL].fillna(0)
y_workout = df_aug['workout_type_label']
y_intensity = df_aug['intensity_label']

print(f'X shape: {X.shape}')
print(f'y_workout classes: {y_workout.unique()}')
print(f'y_intensity classes: {y_intensity.unique()}')

In [ ]:
# ── 4. Train/Test Split (stratified) ──
X_train, X_test, yw_train, yw_test, yi_train, yi_test = train_test_split(
    X, y_workout, y_intensity,
    test_size=0.2, random_state=42, stratify=y_workout
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'\nTrain workout_type dist: {dict(yw_train.value_counts())}')
print(f'Test  workout_type dist: {dict(yw_test.value_counts())}')

In [ ]:
# ── 5. RobustScaler (i-MRI 2024 benchmark wins) ──
NUMERIC_COLS = ['BMI', 'Age', 'session_minutes', 'bmr', 'tdee', 'ffmi',
                'bmi_x_age', 'activity_x_duration', 'bpm_intensity_ratio',
                'calorie_efficiency']

scaler = RobustScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[NUMERIC_COLS] = scaler.fit_transform(X_train[NUMERIC_COLS])
X_test_scaled[NUMERIC_COLS] = scaler.transform(X_test[NUMERIC_COLS])

print('✅ RobustScaler fitted & applied')
print(f'Scaler median: {dict(zip(NUMERIC_COLS, scaler.center_.round(2)))}')
print(f'Scaler IQR:    {dict(zip(NUMERIC_COLS, scaler.scale_.round(2)))}')

In [ ]:
# ── 6. SMOTEENN untuk Class Imbalance (Springer 2024: paling robust) ──

print('Before SMOTEENN:')
print(f'  Workout type: {dict(yw_train.value_counts())}')

# Untuk workout_type (target 1)
smote_enn = SMOTEENN(
    smote=SMOTE(random_state=42, k_neighbors=5),
    enn=EditedNearestNeighbours(n_neighbors=3),
    random_state=42
)
X_train_balanced, yw_train_balanced = smote_enn.fit_resample(X_train_scaled, yw_train)

print('\nAfter SMOTEENN (workout_type):')
print(f'  Workout type: {dict(pd.Series(yw_train_balanced).value_counts())}')
print(f'  Total rows: {len(X_train_balanced)} (was {len(X_train_scaled)})')

# Untuk intensity (target 2) — separate balancing
smote_enn_int = SMOTEENN(
    smote=SMOTE(random_state=42, k_neighbors=5),
    random_state=42
)
X_train_balanced_int, yi_train_balanced = smote_enn_int.fit_resample(X_train_scaled, yi_train)

print('\nAfter SMOTEENN (intensity):')
print(f'  Intensity: {dict(pd.Series(yi_train_balanced).value_counts())}')

In [ ]:
# ── 7. Simpan Semua Output ──

# Scaled (no balance) — untuk model yang punya class_weight handling
X_train_scaled.to_parquet('output/preprocessed/X_train_scaled.parquet', index=False)
X_test_scaled.to_parquet('output/preprocessed/X_test_scaled.parquet', index=False)
yw_train.to_frame().to_parquet('output/preprocessed/yw_train.parquet', index=False)
yw_test.to_frame().to_parquet('output/preprocessed/yw_test.parquet', index=False)
yi_train.to_frame().to_parquet('output/preprocessed/yi_train.parquet', index=False)
yi_test.to_frame().to_parquet('output/preprocessed/yi_test.parquet', index=False)

# Balanced (SMOTEENN) — untuk XGBoost training
pd.DataFrame(X_train_balanced, columns=X_train.columns).to_parquet(
    'output/preprocessed/X_train_balanced.parquet', index=False)
pd.DataFrame({'workout_type_label': yw_train_balanced}).to_parquet(
    'output/preprocessed/yw_train_balanced.parquet', index=False)

pd.DataFrame(X_train_balanced_int, columns=X_train.columns).to_parquet(
    'output/preprocessed/X_train_balanced_intensity.parquet', index=False)
pd.DataFrame({'intensity_label': yi_train_balanced}).to_parquet(
    'output/preprocessed/yi_train_balanced.parquet', index=False)

# Scaler
joblib.dump(scaler, 'output/preprocessed/scaler.pkl')

# Metadata
metadata = {
    'feature_cols': FEATURE_FINAL,
    'numeric_cols_scaled': NUMERIC_COLS,
    'workout_type_map': {0: 'FLEXIBILITY', 1: 'CARDIO', 2: 'STRENGTH', 3: 'HIIT', 4: 'REST'},
    'intensity_map': {1: 'LOW', 2: 'MID', 3: 'HIGH'},
    'preprocessing_stack': {
        'scaler': 'RobustScaler',
        'balancing': 'SMOTEENN',
        'feature_engineering': ['BMR_Mifflin_St_Jeor', 'TDEE', 'FFMI',
                                 'bmi_x_age', 'activity_x_duration',
                                 'bpm_intensity_ratio', 'calorie_efficiency'],
    },
    'n_train_original': len(X_train),
    'n_train_balanced_workout': int(len(X_train_balanced)),
    'n_train_balanced_intensity': int(len(X_train_balanced_int)),
    'n_test': len(X_test),
    'random_seed': 42,
}
with open('output/preprocessed/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Preprocessing selesai. Files saved:')
import os
for f in sorted(os.listdir('output/preprocessed')):
    size = os.path.getsize(f'output/preprocessed/{f}') / 1024
    print(f'  output/preprocessed/{f:40s} ({size:.1f} KB)')